# <center> How to apply TICOI on a cube stored on the ITS_LIVE Amazon S3 Cloud ? </center>


In [19]:
import warnings
import os
from ticoi.core import process, process_blocks_refine, save_cube_parameters
from ticoi.cube_data_classxr import CubeDataClass
from ticoi.cube_writer import CubeResultsWriter
from ticoi.utils import find_granule_by_point

warnings.filterwarnings("ignore")

In [20]:
#avoid asyncio error in jupyternotebook
import nest_asyncio
nest_asyncio.apply()

## 1) Parameters

Path of the input and output data (to change)

In [21]:
i, j = -140.84746241412898, 60.08936753843976 # coordinates of the central pixel of the point
cube_name = find_granule_by_point([i, j]) # function to find the associated ITS_LIVE url
result_fn = "Lowell_example"  # Name of the netCDF file to be created to store the results

In [22]:
print(cube_name)

['s3://its-live-data/datacubes/v2-updated-october2024/N60W140/ITS_LIVE_vel_EPSG3413_G0120_X-3250000_Y350000.zarr']


Main parameters (to change)

In [23]:
## ----------------------- saving options --------------------- ##
#Output to save or not
save = False  # If True, save TICOI results to a netCDF file
save_mean_velocity = False  # Save a .tiff file with the mean resulting velocities, as an example

# What results must be returned from TICOI processing (can be a list of both)
#   - 'invert' for the results of the inversion, corresponding to Cumulative displacement time series
#   - 'interp' for the results of the interpolation
returned = ["invert", "interp"]

## ----------------------- TICOI main parameters --------------------- ##
# We advice the user to change only the following parameter, the other parameters stored in a dictionary can be kept as it is for a first use
regu = "1accelnotnull"  # Regularization method.s to be use: 1 minimize the acceleration, '1accelnotnull' minimize the distance with an apriori on the acceleration computed over a spatio-temporal filtering of the cube
coef = 100  # Regularization coefficient.s to be used
delete_outlier = None  # Delete data, check possible parameters in README_possible_parameters

nb_cpu = 12  # Number of CPU to be used for parallelization
block_size = 0.5  # Maximum sub-block size (in GB) for the 'block_process' TICOI processing method

path_save = os.path.join(
    os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..")),
    "examples",
    "results",
    "cube",
) # path where to store the results

Dictionary to store the different parameters (no need to change it)

In [24]:
load_kwargs = {
    "chunks": {},
    "proj": "EPSG:4326",  # EPSG system of the given coordinates
    "buffer":[i,j,0.01]
}

preData_kwargs = {
    "delete_outliers": delete_outlier,  # Delete data with a poor quality indicator (if int), or with aberrant direction ('vvc_angle')
    "regu": regu,  # Regularization method.s to be used (for each flag if flag is not None) : 1 minimize the acceleration, '1accelnotnull' minize the distance with an apriori on the acceleration computed over a spatio-temporal filtering of the cube
    "proj": "EPSG:3413",  # EPSG system of the given coordinates
}
## ---------------- Inversion and interpolation parameters ----------------- ##
inversion_kwargs = {
    "coef": coef,  # Regularization coefficient.s to be used (for each flag if flag is not None)
    "detect_temporal_decorrelation": True,  # If True, the first inversion will use only velocity observations with small temporal baselines, to detect temporal decorelation
    "result_quality": "X_contribution",  # Criterium used to evaluate the quality of the results ('Norm_residual', 'X_contribution')
    "proj": "EPSG:3413","path_save":path_save
}

## 2) Cube loading

In [ ]:
# Load the first cube
cube = CubeDataClass()
cube.load(cube_name, **load_kwargs)

## 2) Cube processing

In [25]:
#Prepare interpolation dates
first_date_interpol, last_date_interpol = cube.prepare_interpolation_date()
inversion_kwargs.update({"first_date_interpol": first_date_interpol, "last_date_interpol": last_date_interpol})

# TICOI processing
# The data cube is subdivided in smaller cubes computed one after the other in a synchronous manner (uses async)
# TICOI computation is then parallelized among those cubes
result = process_blocks_refine(
    cube,
    nb_cpu=nb_cpu,
    block_size=block_size,
    preData_kwargs=preData_kwargs,
    inversion_kwargs=inversion_kwargs,
    returned=returned
)

20 11
[Block process] Cube size smaller than 0.5GB, no need to divide
[Block process] Processing block 1/1
Block 1 loaded in 24.65 s





  0%|          | 0/220 [00:00<?, ?it/s]


  5%|▌         | 12/220 [00:00<00:03, 58.86it/s]


 11%|█         | 24/220 [00:12<01:55,  1.70it/s]


 16%|█▋        | 36/220 [00:20<01:56,  1.58it/s]


 22%|██▏       | 48/220 [00:29<01:55,  1.49it/s]


 27%|██▋       | 60/220 [00:37<01:49,  1.46it/s]


 33%|███▎      | 72/220 [00:47<01:47,  1.38it/s]


 38%|███▊      | 84/220 [00:56<01:42,  1.33it/s]


 44%|████▎     | 96/220 [01:06<01:35,  1.29it/s]


 49%|████▉     | 108/220 [01:17<01:31,  1.22it/s]


 55%|█████▍    | 120/220 [01:27<01:21,  1.22it/s]


 60%|██████    | 132/220 [01:38<01:13,  1.19it/s]


 65%|██████▌   | 144/220 [01:48<01:03,  1.19it/s]


 71%|███████   | 156/220 [01:54<00:48,  1.33it/s]


 76%|███████▋  | 168/220 [02:04<00:40,  1.29it/s]


 82%|████████▏ | 180/220 [02:12<00:29,  1.34it/s]


 87%|████████▋ | 192/220 [02:22<00:21,  1.32it/s]


 93%|█████████▎| 204/220 [02:31<00:12,  1.33it/s]


100%|██████████| 220/220 [02:38<00:00,  1.39it/s]


## 3) Cube saving

In [26]:
if save:
    if "invert" in returned:
        source, sensor = save_cube_parameters(cube, load_kwargs, preData_kwargs, inversion_kwargs, returned="invert")
    if "interp" in returned:
        source_interp, sensor = save_cube_parameters(
            cube, load_kwargs, preData_kwargs, inversion_kwargs, returned="interp"
        )

    several = isinstance(returned, list) and len(returned) >= 2
    writer = CubeResultsWriter(cube)

    if "invert" in returned:
        cube_invert = writer.write_result_tico(
            result["invert"] if several else result,
            source,
            sensor,
            result_quality=inversion_kwargs["result_quality"],
            filename=f"{result_fn}_invert1" if several else result_fn,
            savepath=path_save if save else None,
            verbose=inversion_kwargs["verbose"],
        )
    if "interp" in returned:
        cube_interp = writer.write_result_ticoi(
            result["interp"] if several else result,
            source_interp,
            sensor,
            result_quality=inversion_kwargs["result_quality"],
            filename=f"{result_fn}_interp1" if several else result_fn,
            savepath=path_save if save else None,
            verbose=inversion_kwargs["verbose"],
        )

## 4) Temporal average of the cube (optional)

In [27]:
# Plot the mean velocity as an example
if save_mean_velocity and cube_interp is not None:
    cube_interp.average_cube(return_format="geotiff", return_variable=["vv"], save=True, path_save=path_save)